In [1]:
import pandas as pd
import html
import re

df = pd.read_csv('../../data/raw/reddit_raw/reddit_ai_wikipedia_posts.csv')
df['date'] = pd.to_datetime(df['date'])
print(f'starting: {len(df)} posts')

starting: 1764 posts


In [2]:
before = len(df)
df = df.sort_values('score', ascending=False).drop_duplicates(subset=['title', 'subreddit']).reset_index(drop=True)
print(f'dropped {before - len(df)} same-subreddit title duplicates')

dropped 3 same-subreddit title duplicates


In [3]:
url_pattern = re.compile(r'^https?://\S+\s*$')
img_mask = df['text'].notna() & df['text'].apply(lambda x: bool(url_pattern.match(str(x).strip())))
df.loc[img_mask, 'text'] = None
print(f'nulled {img_mask.sum()} image/URL-only text fields')

nulled 15 image/URL-only text fields


In [6]:
for col in ['text', 'combined_text', 'title']:
    df[col] = df[col].apply(lambda x: html.unescape(str(x)) if pd.notnull(x) else x)
print('decoded HTML entities')

df['combined_text'] = df['title'].fillna('') + ' ' + df['text'].fillna('')
df['combined_text'] = df['combined_text'].str.strip()

decoded HTML entities


In [7]:
ai_kw = ['ai', 'chatgpt', 'gpt', 'llm', 'language model', 'artificial intelligence',
         'openai', 'gemini', 'claude', 'machine learning', 'neural', 'deep learning']
wiki_kw = ['wikipedia', 'wiki']

combined_lower = df['combined_text'].str.lower()
has_wiki = combined_lower.apply(lambda x: any(k in x for k in wiki_kw))
has_ai = combined_lower.apply(lambda x: any(k in x for k in ai_kw))

before = len(df)
df = df[has_wiki | has_ai].reset_index(drop=True)
print(f'dropped {before - len(df)} posts with no AI or Wikipedia keyword')

dropped 170 posts with no AI or Wikipedia keyword


In [8]:
df['zero_score'] = df['score'] == 0
print(f'flagged {df["zero_score"].sum()} zero-score posts')

flagged 284 zero-score posts


In [10]:
df['title_word_count'] = df['title'].fillna('').apply(lambda x: len(str(x).split()))

print(f'\nfinal: {len(df)} posts')
print('\nby subreddit:')
print(df['subreddit'].value_counts())
print('\nby period:')
print(df['period'].value_counts())
print(f'\nnull text: {df["text"].isnull().sum()} ({df["text"].isnull().mean()*100:.1f}%)')

df.to_csv('../../data/raw/reddit_raw/reddit_ai_wikipedia_posts.csv', index=False)
print('\nsaved cleaned file')



final: 1591 posts

by subreddit:
subreddit
ChatGPT            450
MachineLearning    293
singularity        240
wikipedia          205
OpenAI             177
artificial         162
technology          64
Name: count, dtype: int64

by period:
period
post-ChatGPT    1298
pre-ChatGPT      293
Name: count, dtype: int64

null text: 387 (24.3%)

saved cleaned file


In [11]:
df

,subreddit,title,text,score,date,num_comments,url,permalink,id,title_word_count,combined_text,period,zero_score
0,technology,"Wikipedia turns 25, still boasting zero ads an...",NaN,62607,2026-01-18 20:58:29,893,https://www.pcgamer.com/gaming-industry/wikipe...,https://reddit.com/r/technology/comments/1qgk4...,1qgk4it,24,"Wikipedia turns 25, still boasting zero ads an...",post-ChatGPT,False
1,technology,New survey suggests the vast majority of iPhon...,NaN,8327,2025-03-08 14:26:48,516,https://www.techradar.com/phones/new-survey-su...,https://reddit.com/r/technology/comments/1j6i9...,1j6i9jb,20,New survey suggests the vast majority of iPhon...,post-ChatGPT,False
2,technology,Wikipedia urges AI companies to use its paid A...,NaN,7659,2025-11-10 18:36:09,184,https://techcrunch.com/2025/11/10/wikipedia-ur...,https://reddit.com/r/technology/comments/1otlw...,1otlwta,14,Wikipedia urges AI companies to use its paid A...,post-ChatGPT,False
3,wikipedia,In a minefield of glitchy AI search and social...,NaN,6607,2025-01-14 22:40:33,116,https://edition.cnn.com/2025/01/14/business/wi...,https://reddit.com/r/wikipedia/comments/1i1inh...,1i1inhg,21,In a minefield of glitchy AI search and social...,post-ChatGPT,False
4,technology,Wikipedia Says AI Is Causing a Dangerous Decli...,NaN,4658,2025-10-17 01:33:39,166,https://www.404media.co/wikipedia-says-ai-is-c...,https://reddit.com/r/technology/comments/1o8od...,1o8odd4,33,Wikipedia Says AI Is Causing a Dangerous Decli...,post-ChatGPT,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1586,MachineLearning,[P]I turned Elon Musk's face into a decision b...,I've seen examples of 2d decision boundaries t...,0,2024-03-29 08:44:07,15,https://www.reddit.com/r/MachineLearning/comme...,https://reddit.com/r/MachineLearning/comments/...,1bqkcor,9,[P]I turned Elon Musk's face into a decision b...,post-ChatGPT,True
1587,MachineLearning,[D] Is the competition/cooperation between sym...,I have found that comprehensive overviews of ...,0,2022-02-17 22:56:56,8,https://www.reddit.com/r/MachineLearning/comme...,https://reddit.com/r/MachineLearning/comments/...,sv1wuq,28,[D] Is the competition/cooperation between sym...,pre-ChatGPT,True
1588,MachineLearning,[D] Best approach for noisy language detection,"I need to predict the language (e.g. English, ...",0,2021-11-04 22:12:36,0,https://www.reddit.com/r/MachineLearning/comme...,https://reddit.com/r/MachineLearning/comments/...,qmw43e,7,[D] Best approach for noisy language detection...,pre-ChatGPT,True
1589,wikipedia,Help me find wikipedia article?,Trying to find a wikipedia article/book synops...,0,2026-02-27 19:28:48,2,https://www.reddit.com/r/wikipedia/comments/1r...,https://reddit.com/r/wikipedia/comments/1rggcq...,1rggcqp,5,Help me find wikipedia article? Trying to find...,post-ChatGPT,True
